In [ ]:
import os
import json
import random
import hashlib
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from PIL import Image, UnidentifiedImageError
from dotenv import load_dotenv

env_path = Path('../.env')
load_dotenv(dotenv_path=env_path)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# ── Paths ──────────────────────────────────────────────
HAM_DIR    = Path('../data/data_ham10000')
DERM_DIR   = Path('../data/derm7pt_data/release_v0')
OUTPUT_DIR = Path('../results/ph03_outputs')
SPLIT_DIR  = Path('../data/splits')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SPLIT_DIR.mkdir(parents=True, exist_ok=True)

# ── Cấu hình chuẩn hóa ─────────────────────────────────
IMG_SIZE = 224                    # dùng chung ResNet & ViT
MEAN     = [0.485, 0.456, 0.406]  # ImageNet mean
STD      = [0.229, 0.224, 0.225]  # ImageNet std

print('Setup done.')
print(f'IMG_SIZE = {IMG_SIZE}, MEAN = {MEAN}, STD = {STD}')

Setup done. IMG_SIZE = 224


In [42]:
# ── Load metadata HAM10000 ─────────────────────────────
ham_csv = HAM_DIR / 'HAM10000_metadata.csv'
df_ham  = pd.read_csv(ham_csv)
print(f'HAM10000 — {len(df_ham):,} dòng, {df_ham.shape[1]} cột')

missing = df_ham.isnull().sum()
print('\nMissing values:')
print(missing[missing > 0] if missing.any() else 'Không có missing values')

HAM10000 — 10,015 dòng, 8 cột

Missing values:
age    57
dtype: int64


In [43]:
ham_img_dirs = [
    HAM_DIR / 'HAM10000_images_part_1',
    HAM_DIR / 'HAM10000_images_part_2',
]

def find_ham_image(image_id, img_dirs):
    for d in img_dirs:
        p = d / f'{image_id}.jpg'
        if p.exists():
            return p
    return None

df_ham['img_path'] = df_ham['image_id'].apply(lambda x: find_ham_image(x, ham_img_dirs))
missing_imgs = df_ham['img_path'].isna().sum()
print(f'Ảnh tìm thấy : {len(df_ham) - missing_imgs:,} / {len(df_ham):,}')
print(f'Ảnh bị thiếu : {missing_imgs}')

Ảnh tìm thấy : 10,015 / 10,015
Ảnh bị thiếu : 0


In [44]:
corrupt_ham = []
for path in df_ham['img_path'].dropna():
    try:
        with Image.open(path) as img:
            img.verify()
    except Exception:
        corrupt_ham.append(str(path))

print(f'Ảnh corrupt: {len(corrupt_ham)}')
if corrupt_ham:
    print('Danh sách ảnh lỗi:', corrupt_ham[:5])

Ảnh corrupt: 0


In [45]:
def md5_hash(path):
    with open(path, 'rb') as f:
        return hashlib.md5(f.read()).hexdigest()

sample_paths = df_ham['img_path'].dropna().sample(min(500, len(df_ham)), random_state=SEED).tolist()
hashes    = [md5_hash(p) for p in sample_paths]
dup_count = len(hashes) - len(set(hashes))
print(f'Trùng ảnh trong sample 500: {dup_count}')

Trùng ảnh trong sample 500: 0


In [46]:
label_dist_ham = df_ham['dx'].value_counts()
print('Phân phối nhãn HAM10000:')
print(label_dist_ham)

fig, ax = plt.subplots(figsize=(8, 4))
label_dist_ham.plot(kind='bar', ax=ax, color='steelblue')
ax.set_title('Phân phối nhãn — HAM10000')
ax.set_xlabel('Label'); ax.set_ylabel('Số ảnh')
plt.tight_layout()
fig.savefig(OUTPUT_DIR / 'ham10000_label_dist.png', dpi=100)
plt.close()
print('Đã lưu ham10000_label_dist.png')

Phân phối nhãn HAM10000:
dx
nv       6705
mel      1113
bkl      1099
bcc       514
akiec     327
vasc      142
df        115
Name: count, dtype: int64
Đã lưu ham10000_label_dist.png


## Load metadata Derm7pt

In [47]:
derm_csv = DERM_DIR / 'meta' / 'meta.csv'
df_derm  = pd.read_csv(derm_csv)
print(f'Derm7pt — {len(df_derm):,} dòng, {df_derm.shape[1]} cột')
print('\nDanh sách cột:', df_derm.columns.tolist())
print('\nCác giá trị cột diagnosis:')
print(df_derm['diagnosis'].value_counts())

Derm7pt — 1,011 dòng, 19 cột

Danh sách cột: ['case_num', 'diagnosis', 'seven_point_score', 'pigment_network', 'streaks', 'pigmentation', 'regression_structures', 'dots_and_globules', 'blue_whitish_veil', 'vascular_structures', 'level_of_diagnostic_difficulty', 'elevation', 'location', 'sex', 'management', 'clinic', 'derm', 'case_id', 'notes']

Các giá trị cột diagnosis:
diagnosis
clark nevus                     399
melanoma (less than 0.76 mm)    102
reed or spitz nevus              79
melanoma (in situ)               64
melanoma (0.76 to 1.5 mm)        53
seborrheic keratosis             45
basal cell carcinoma             42
dermal nevus                     33
vascular lesion                  29
blue nevus                       28
melanoma (more than 1.5 mm)      28
lentigo                          24
dermatofibroma                   20
congenital nevus                 17
melanosis                        16
combined nevus                   13
miscellaneous                     8
recu

In [48]:
DERM_LABEL_MAP = {
    # ── NEVUS group ────────────────────────────────────
    'clark nevus'                  : 'nevus',
    'reed or spitz nevus'          : 'nevus',
    'dermal nevus'                 : 'nevus',
    'blue nevus'                   : 'nevus',
    'congenital nevus'             : 'nevus',
    'combined nevus'               : 'nevus',
    'recurrent nevus'              : 'nevus',
    # ── MELANOMA group ─────────────────────────────────
    'melanoma'                     : 'mel',
    'melanoma (in situ)'           : 'mel',
    'melanoma (less than 0.76 mm)' : 'mel',
    'melanoma (0.76 to 1.5 mm)'    : 'mel',
    'melanoma (more than 1.5 mm)'  : 'mel',
    'melanoma metastasis'          : 'mel',
    # ── SEBORRHEIC KERATOSIS ───────────────────────────
    'seborrheic keratosis'         : 'sk',
    # ── BASAL CELL CARCINOMA ───────────────────────────
    'basal cell carcinoma'         : 'bcc',
    # ── MISC group ─────────────────────────────────────
    'miscellaneous'                : 'misc',
    'vascular lesion'              : 'misc',
    'lentigo'                      : 'misc',
    'dermatofibroma'               : 'misc',
    'melanosis'                    : 'misc',
}

df_derm['label'] = (
    df_derm['diagnosis']
    .str.lower()
    .str.strip()
    .map(DERM_LABEL_MAP)
)

unmapped = df_derm['label'].isna().sum()
print(f'Nhãn không map được: {unmapped}')
if unmapped:
    print('⚠️ Giá trị chưa map:')
    print(df_derm.loc[df_derm['label'].isna(), 'diagnosis'].value_counts())
else:
    print('✅ Tất cả nhãn đã được map!')
print('\nPhân phối 5 lớp sau mapping:')
print(df_derm['label'].value_counts())

Nhãn không map được: 0
✅ Tất cả nhãn đã được map!

Phân phối 5 lớp sau mapping:
label
nevus    575
mel      252
misc      97
sk        45
bcc       42
Name: count, dtype: int64


In [49]:
DERM_IMG_DIR = DERM_DIR / 'images'

def find_derm_image(row):
    img_name = str(row.get('derm', '')).strip()
    if not img_name:
        return None
    p = DERM_IMG_DIR / img_name
    return p if p.exists() else None

df_derm['img_path'] = df_derm.apply(find_derm_image, axis=1)
missing_derm = df_derm['img_path'].isna().sum()
print(f'Derm7pt ảnh tìm thấy : {len(df_derm) - missing_derm:,} / {len(df_derm):,}')
print(f'Ảnh bị thiếu trên disk: {missing_derm}')

Derm7pt ảnh tìm thấy : 1,011 / 1,011
Ảnh bị thiếu trên disk: 0


In [50]:
df_derm_clean = df_derm.dropna(subset=['img_path', 'label']).copy()
print(f'Sau khi clean: {len(df_derm_clean):,} ảnh hợp lệ')
print('\nPhân phối nhãn sau clean:')
print(df_derm_clean['label'].value_counts())

Sau khi clean: 1,011 ảnh hợp lệ

Phân phối nhãn sau clean:
label
nevus    575
mel      252
misc      97
sk        45
bcc       42
Name: count, dtype: int64


In [51]:
META_DIR  = DERM_DIR / 'meta'

train_idx = pd.read_csv(META_DIR / 'train_indexes.csv')
val_idx   = pd.read_csv(META_DIR / 'valid_indexes.csv')
test_idx  = pd.read_csv(META_DIR / 'test_indexes.csv')

train_set = set(train_idx['indexes'].tolist())
val_set   = set(val_idx['indexes'].tolist())
test_set  = set(test_idx['indexes'].tolist())

print(f'Số index — Train: {len(train_set)} | Val: {len(val_set)} | Test: {len(test_set)}')

Số index — Train: 413 | Val: 203 | Test: 395


In [52]:
df_train = df_derm.loc[df_derm.index.isin(train_set)].dropna(subset=['img_path', 'label']).copy()
df_val   = df_derm.loc[df_derm.index.isin(val_set)].dropna(subset=['img_path', 'label']).copy()
df_test  = df_derm.loc[df_derm.index.isin(test_set)].dropna(subset=['img_path', 'label']).copy()

total = len(df_train) + len(df_val) + len(df_test)
print(f'Train : {len(df_train):,}  ({len(df_train)/total:.1%})')
print(f'Val   : {len(df_val):,}  ({len(df_val)/total:.1%})')
print(f'Test  : {len(df_test):,}  ({len(df_test)/total:.1%})')
print(f'Tổng  : {total:,}')

assert len(set(df_train.index) & set(df_val.index))  == 0, 'Train/Val overlap!'
assert len(set(df_train.index) & set(df_test.index)) == 0, 'Train/Test overlap!'
assert len(set(df_val.index)   & set(df_test.index)) == 0, 'Val/Test overlap!'
print('\n✅ Không có overlap giữa các split')

df_train.to_csv(SPLIT_DIR / 'derm7pt_train.csv', index=False)
df_val.to_csv(SPLIT_DIR   / 'derm7pt_val.csv',   index=False)
df_test.to_csv(SPLIT_DIR  / 'derm7pt_test.csv',  index=False)
print(f'Đã lưu split vào {SPLIT_DIR}')

Train : 413  (40.9%)
Val   : 203  (20.1%)
Test  : 395  (39.1%)
Tổng  : 1,011

✅ Không có overlap giữa các split
Đã lưu split vào ..\data\splits


In [53]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=True)
for ax, (df_split, title) in zip(axes, [(df_train,'Train'), (df_val,'Val'), (df_test,'Test')]):
    df_split['label'].value_counts().sort_index().plot(kind='bar', ax=ax, color='teal')
    ax.set_title(title)
    ax.set_xlabel('Label')
    ax.set_ylabel('Count')
plt.suptitle('Phân phối nhãn theo split — Derm7pt', y=1.02)
plt.tight_layout()
fig.savefig(OUTPUT_DIR / 'derm7pt_split_dist.png', dpi=100)
plt.close()
print('Đã lưu derm7pt_split_dist.png')

Đã lưu derm7pt_split_dist.png


In [57]:
pip install albumentations

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [59]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())  # False là bình thường với CPU

OSError: [WinError 1114] A dynamic link library (DLL) initialization routine failed. Error loading "c:\Users\ADMIN\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\lib\c10.dll" or one of its dependencies.